# Finding the Best Delivery Promise for Rosa's Pizza
## MMGMT 722 — Assignment 1

This notebook examines Rosa's delivery performance under the current
45-minute promise and explores which promised delivery times produce
the highest net profit for each zone and time block.

In [1]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [3]:
print("Delivery zones:", ZONES)
print("Time blocks:", TIME_BLOCKS)
print("Cost assumptions:", COSTS)
print("Current delivery promise:", PROMISE, "minutes")

Delivery zones: ['Central', 'North', 'Far West']
Time blocks: ['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
Cost assumptions: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
Current delivery promise: 45 minutes


## Exploring the delivery simulator

I use the provided starter package to simulate four weeks of deliveries
in Far West during Friday/Saturday evenings with a 45-minute promise.
Each value represents one order's delivery time in minutes.

I use seed=1 to make the simulation reproducible.

In [4]:
times = delivery_times("Far West", "Fri/Sat eve", PROMISE, seed=1)

print("Number of orders:", len(times))
print("First 10 delivery times:", times[:10])
print("Average delivery time:", round(times.mean(), 1), "minutes")

Number of orders: 210
First 10 delivery times: [46.6 53.7 46.4 28.4 55.1 48.  35.8 50.  46.9 45.9]
Average delivery time: 42.9 minutes


## Initial delivery results

The simulation generated 210 orders with an average delivery time of
42.9 minutes. However, 77 orders arrived after the 45-minute promise,
giving a late-order percentage of 36.67%.

Although the average delivery time was below the promise, more than
one-third of orders were late. This shows why the average alone does
not fully describe whether Rosa meets her delivery promise.

In [5]:
late_orders = times > PROMISE
number_late = np.sum(late_orders)
late_percentage = number_late / len(times) * 100

print("First 10 late-order checks:", late_orders[:10])
print("Number of late orders:", number_late)
print("Percentage of late orders:", round(late_percentage, 2), "%")

First 10 late-order checks: [ True  True  True False  True  True False  True  True  True]
Number of late orders: 77
Percentage of late orders: 36.67 %


## Part I(a): Percentage of late orders

In [6]:
def late_rate(zone, time_block, promise):
    if zone == "all":
        selected_zones = ZONES
    else:
        selected_zones = [zone]

    if time_block == "all":
        selected_blocks = TIME_BLOCKS
    else:
        selected_blocks = [time_block]

    total_orders = 0
    total_late = 0

    for z in selected_zones:
        for block in selected_blocks:
            times = delivery_times(z, block, promise, seed=1)
            total_orders = total_orders + len(times)
            total_late = total_late + np.sum(times > promise)

    return total_late / total_orders * 100

In [7]:
print("Far West, Fri/Sat eve:",
      round(late_rate("Far West", "Fri/Sat eve", 45), 2), "%")

print("All zones, Lunch:",
      round(late_rate("all", "Lunch", 45), 2), "%")

print("Far West, all time blocks:",
      round(late_rate("Far West", "all", 45), 2), "%")

print("All zones, all time blocks:",
      round(late_rate("all", "all", 45), 2), "%")

Far West, Fri/Sat eve: 36.67 %
All zones, Lunch: 3.33 %
Far West, all time blocks: 15.61 %
All zones, all time blocks: 8.29 %


### Results under the 45-minute promise

The late-order percentage was 36.67% in Far West on Friday/Saturday
evenings, compared with 15.61% across all time blocks in Far West.
Across all zones, the lunch late rate was 3.33%, while the overall
late rate was 8.29%.

These results show that the overall rate can hide a problem within
a particular zone and time block. Next, I rank all 12 combinations
to identify those with the highest late rates.

## Part I(b): Ranking by late percentage

In [8]:
import pandas as pd

results = []

for zone in ZONES:
    for block in TIME_BLOCKS:
        rate = late_rate(zone, block, PROMISE)

        results.append({
            "Zone": zone,
            "Time block": block,
            "Late percentage": rate
        })

late_ranking = pd.DataFrame(results)

late_ranking = late_ranking.sort_values(
    "Late percentage", ascending=False
)

late_ranking.round(2)

,Zone,Time block,Late percentage
10,Far West,Fri/Sat eve,36.67
6,North,Fri/Sat eve,16.67
9,Far West,Weekday eve,11.18
2,Central,Fri/Sat eve,6.84
5,North,Weekday eve,6.00
11,Far West,Other,4.69
1,Central,Weekday eve,4.24
4,North,Lunch,3.68
0,Central,Lunch,3.46
3,Central,Other,2.63


### Interpretation of the late-percentage ranking

Under the 45-minute promise, Far West on Friday/Saturday evenings
had the highest simulated late percentage (36.67%), followed by North
on Friday/Saturday evenings (16.67%) and Far West on weekday evenings
(11.18%).

Friday/Saturday evenings had the highest late percentage within
each zone. This suggests that these time blocks deserve particular
attention when evaluating the delivery promise.

The lowest late percentages were in North during Other time blocks
and Far West during Lunch, both at 2.50%.

## Part I(c): Average delivery time

I create a reusable function to calculate the average delivery time
for a specific zone and time block, or across all zones or time blocks.

For combined groups, I divide the total delivery time by the total
number of orders, so each order receives equal weight.

In [9]:
def average_delivery_time(zone, time_block, promise):
    if zone == "all":
        selected_zones = ZONES
    else:
        selected_zones = [zone]

    if time_block == "all":
        selected_blocks = TIME_BLOCKS
    else:
        selected_blocks = [time_block]

    total_orders = 0
    total_minutes = 0

    for z in selected_zones:
        for block in selected_blocks:
            times = delivery_times(z, block, promise, seed=1)

            total_orders = total_orders + len(times)
            total_minutes = total_minutes + np.sum(times)

    return total_minutes / total_orders

In [10]:
print("Far West, Fri/Sat eve:",
      round(average_delivery_time("Far West", "Fri/Sat eve", 45), 2),
      "minutes")

print("All zones, Lunch:",
      round(average_delivery_time("all", "Lunch", 45), 2),
      "minutes")

print("Far West, all time blocks:",
      round(average_delivery_time("Far West", "all", 45), 2),
      "minutes")

print("All zones, all time blocks:",
      round(average_delivery_time("all", "all", 45), 2),
      "minutes")

Far West, Fri/Sat eve: 42.92 minutes
All zones, Lunch: 28.22 minutes
Far West, all time blocks: 35.27 minutes
All zones, all time blocks: 31.19 minutes


### Interpretation of average delivery times

Under the 45-minute promise, the simulated average delivery time was
42.92 minutes in Far West on Friday/Saturday evenings, compared with
35.27 minutes across all time blocks in Far West.

Across all zones, lunch deliveries averaged 28.22 minutes, while the
overall average was 31.19 minutes.

Although the Far West Friday/Saturday evening average was below
45 minutes, 36.67% of orders were late. This shows that average delivery
time alone does not fully describe whether the delivery promise is met.

## Part I(d): Ranking combinations by average delivery time

I calculate the average delivery time for each of the 12 zone and
time-block combinations under the 45-minute promise. I then sort
the results from highest to lowest average delivery time.

In [11]:
average_results = []

for zone in ZONES:
    for block in TIME_BLOCKS:
        average = average_delivery_time(zone, block, PROMISE)

        average_results.append({
            "Zone": zone,
            "Time block": block,
            "Average delivery time (minutes)": average
        })

average_ranking = pd.DataFrame(average_results)

average_ranking = average_ranking.sort_values(
    "Average delivery time (minutes)", ascending=False
)

average_ranking.round(2)

,Zone,Time block,Average delivery time (minutes)
10,Far West,Fri/Sat eve,42.92
6,North,Fri/Sat eve,36.77
9,Far West,Weekday eve,35.04
2,Central,Fri/Sat eve,32.13
5,North,Weekday eve,31.38
8,Far West,Lunch,30.58
11,Far West,Other,30.04
1,Central,Weekday eve,29.03
4,North,Lunch,28.67
7,North,Other,27.66


### Interpretation of the average delivery-time ranking

Under the 45-minute promise, Far West on Friday/Saturday evenings
had the longest simulated average delivery time (42.92 minutes),
followed by North on Friday/Saturday evenings (36.77 minutes) and
Far West on weekday evenings (35.04 minutes).

These combinations also had the three highest late percentages.
Central during Other time blocks had the shortest average delivery
time, at 26.04 minutes.

The two rankings were not identical. For example, Far West during
Lunch had a higher average delivery time than Far West during Other
time blocks, but a lower late percentage. This shows that average
delivery time and late percentage describe different aspects of
delivery performance.

## Part I(e): Late percentage versus average delivery time

Late percentage is more directly relevant when evaluating Rosa’s
45-minute delivery promise. It measures the proportion of orders
that miss the promised time. These late orders can result in refunds
and lost future orders, so they have a direct financial impact.

Average delivery time is useful for understanding delivery speed,
but it can hide late deliveries. For example, Far West on
Friday/Saturday evenings averaged 42.92 minutes, below the
45-minute promise, yet 36.67% of orders were late. Faster deliveries
lower the average without removing the problems experienced by
customers whose orders arrive late.

The rankings in Parts I(b) and I(d) identify the same three highest
combinations: Far West on Friday/Saturday evenings, North on
Friday/Saturday evenings, and Far West on weekday evenings.
However, the rankings differ for other combinations. Far West
during Lunch averaged 30.58 minutes with 2.50% late orders,
while Far West during Other time blocks averaged 30.04 minutes
with 4.69% late orders. A lower average therefore does not
necessarily mean fewer missed promises.

I would use late percentage to assess promise reliability and
average delivery time to understand overall speed. To choose
the best promise, I also need to consider order volume, profit
margin, and the cost of late orders. Simply extending the promise
to reduce lateness could reduce demand and profit.

## Part II(a): Cost per late order

I calculate the cost of one late order as the refund plus the
expected profit lost from future orders:

Cost per late order = refund + (lost future orders × margin per order).

I use the provided COSTS dictionary without changing its values.

In [12]:
def cost_per_late_order(costs):
    return costs["refund"] + costs["churn_orders"] * costs["margin"]

late_cost = cost_per_late_order(COSTS)

print("Cost per late order: $", round(late_cost, 2))

Cost per late order: $ 26.2


## Part II(b): Finding the best delivery promise

I will test delivery promises from 20 to 70 minutes in 5-minute steps.
This range includes the current 45-minute promise and alternatives
both shorter and longer than it.

The range extends below and above the simulated average delivery
times observed in Part I. It allows me to explore the trade-off
between attracting orders with a shorter promise and reducing
late-order costs with a longer promise.

The best promise will be the one with the highest simulated net
profit among the tested values. If the best result is at either
endpoint, I will extend the range to check additional promises.

In [13]:
promises = list(range(20, 71, 5))

print("Delivery promises to test:", promises)

Delivery promises to test: [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]


In [14]:
def best_promise(zone, time_block, promises, costs):
    late_cost = cost_per_late_order(costs)

    best_time = None
    best_profit = None
    results = []

    for promise in promises:
        times = delivery_times(zone, time_block, promise, seed=1)

        number_orders = len(times)
        number_late = np.sum(times > promise)

        profit = (
            number_orders * costs["margin"]
            - number_late * late_cost
        )

        results.append({
            "Promise (minutes)": promise,
            "Number of orders": number_orders,
            "Late orders": number_late,
            "Net profit": profit
        })

        if best_profit is None or profit > best_profit:
            best_time = promise
            best_profit = profit

    return best_time, best_profit, pd.DataFrame(results)

In [15]:
optimal_promise, optimal_profit, profit_table = best_promise(
    "Far West", "Fri/Sat eve", promises, COSTS
)

print("Best delivery promise:", optimal_promise, "minutes")
print("Net profit: $", round(optimal_profit, 2))

profit_table.round(2)


Best delivery promise: 55 minutes
Net profit: $ 1212.4


,Promise (minutes),Number of orders,Late orders,Net profit
0,20,241,239,-4092.8
1,25,239,233,-3953.6
2,30,237,218,-3578.6
3,35,232,179,-2601.8
4,40,224,133,-1468.6
5,45,210,77,-127.4
6,50,188,33,827.4
7,55,158,8,1212.4
8,60,121,2,1036.6
9,65,84,0,756.0


### Interpretation of the best-promise result

For Far West on Friday/Saturday evenings, the best promise among
the values tested from 20 to 70 minutes in 5-minute steps was
55 minutes. Using seed=1 and the provided costs, this produced
158 orders, 8 late orders, and simulated net profit of $1,212.40
over the four-week period.

The current 45-minute promise produced 210 orders and 77 late
orders, resulting in net profit of -$127.40. The 55-minute promise
therefore improved simulated net profit by $1,339.80.

A longer promise reduced late-order costs but also reduced demand.
For example, 65 minutes produced no late orders, but only 84 orders
and $756.00 in net profit. This shows why the best promise must
balance order volume against late-order costs.

The best tested promise was inside the selected range. This result
applies to this zone and time block, the tested promises, and this
simulation seed; it is not a guarantee of future profit.


## Part III: Streamlit App

The Streamlit app reuses the notebook's cost and best-promise
functions through analysis.py. Users can select a zone, time block,
promise range, and cost assumptions.

- GitHub repository: https://github.com/rahima63/rosas-pizza-app
- Deployed app: https://rahima-rosas-pizza.streamlit.app/

For Far West during Fri/Sat eve, using promises from 20 to 70 minutes
in 5-minute steps and the default costs, the app reproduced the
notebook result: a best promise of 55 minutes and simulated net
profit of $1,212.40 over four weeks, using seed=1.

## Appendix: Use of AI Tools

### Tools and how I used them

I used ChatGPT/Codex for step-by-step guidance, explanations of Python concepts, help with errors, and review of my code and results. I also asked for suggestions for improvement and explanations consistent with methods covered in our classes.

I used Claude Code in VS Code to convert the notebook into a Streamlit app. I explicitly asked it to follow .github/skills/notebook-to-streamlit/SKILL.md and reuse the notebook's calculation functions. It helped create analysis.py, app.py, requirements.txt, and .gitignore.

### Examples of prompts used

The following are excerpts or paraphrases of my prompts.

**Code review, debugging, and explanations — ChatGPT/Codex (paraphrased):**

> Please review my Python code and suggest improvements where needed. When I share an error message, help me understand its cause and explain how to fix it.

**Notebook conversion — Claude Code (excerpt):**

> Read and follow .github/skills/notebook-to-streamlit/SKILL.md to convert Rosas_Pizza_Assignment_1.ipynb into a Streamlit app. I am a Python beginner. Work one step at a time. First, create only analysis.py by copying cost_per_late_order() and best_promise() from my notebook, with the imports they need. Keep the same calculation logic and seed=1.

**App development — Claude Code (excerpt):**

> Create only app.py now. Import and reuse the functions from analysis.py without changing their calculation logic.

### Review and verification

I reviewed the generated code with AI explanations and ran the notebook and app. I checked that the app used the official simulator and preserved seed=1.

For Far West during Fri/Sat eve, with promises from 20 to 70 minutes in 5-minute steps and default costs, the app matched the notebook: a best promise of 55 minutes and simulated net profit of $1,212.40 over four weeks.

I also checked that the app displayed an error when the minimum promise exceeded the maximum.

AI helped produce code and explanatory text. I reviewed the outputs before including them in the assignment.